In [ ]:
import pandas as pd
import numpy as np

# Load data
df = pd.read_csv('cosmetic_clean.csv')
print(df.shape)
print(df.head())

(9504, 30)
                                        product_name   website country  \
0  OPP Magnet Apparel Perfume Eau de Parfume (Mag...  Flipkart   India   
1              MONET PASSPORT SPRAY PERFUME GREEN 15  Flipkart   India   
2               Fragrantor's Eau De Perfume (Silver)  Flipkart   India   
3  Pure & Gentle, 98% Pure Glycerin and Natural O...  Flipkart   India   
4                          Pure & Gentle Bathing Bar  Flipkart   India   

  category subcategory                                         title_href  \
0     body     perfume  https://www.amazon.in/OPP-Apparel-Perfume-Parf...   
1     body     perfume  https://www.amazon.in/Monet-Passport-Spray-Per...   
2     body     perfume  https://www.amazon.in/Fragrantors-Eau-De-Perfu...   
3     body        soap  https://www.flipkart.com/pears-pure-gentle-98-...   
4     body        soap  https://www.flipkart.com/pears-pure-gentle-bat...   

   price         brand ingredients    form  ... size_unit size_ml_est  \
0  189.0

Cek Kondisi Awal (Data Profiling)

In [ ]:
# Cek info tipe data dan missing values
df.info()

# Cek berapa banyak data kosong (NaN) di tiap kolom
print(df.isnull().sum())

# Cek data duplikat
print("Jumlah duplikat:", df.duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9504 entries, 0 to 9503
Data columns (total 30 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   product_name        9504 non-null   object 
 1   website             9504 non-null   object 
 2   country             9504 non-null   object 
 3   category            9504 non-null   object 
 4   subcategory         9504 non-null   object 
 5   title_href          9504 non-null   object 
 6   price               9349 non-null   float64
 7   brand               9504 non-null   object 
 8   ingredients         5836 non-null   object 
 9   form                9504 non-null   object 
 10  type                7479 non-null   object 
 11  color               7843 non-null   object 
 12  size                7796 non-null   object 
 13  rating              9241 non-null   object 
 14  noofratings         9503 non-null   object 
 15  rating_num          9233 non-null   float64
 16  review

Membersihkan Kolom Kategorikal (Teks)

In [ ]:
# 1. Membersihkan karakter aneh di kolom brand (ganti encoding error)
# ganti string yang mengandung 'ï¿½' menjadi teks yang lebih bersih
df['brand'] = df['brand'].str.replace('ï¿½', '', regex=False) # Hapus karakter aneh
df['brand'] = df['brand'].str.replace("L'Oral", "L'Oreal", regex=False) # Perbaiki typo
df['brand'] = df['brand'].str.replace("LAKM", "Lakme", regex=False)
df['brand'] = df['brand'].str.strip() # Hapus spasi di awal/akhir
df['brand'] = df['brand'].fillna('Unknown') # Isi brand kosong dengan 'Unknown'

# 2. Rapihin kolom Category & Subcategory (jadikan huruf kecil semua biar konsisten)
df['category'] = df['category'].str.lower().str.strip()
df['subcategory'] = df['subcategory'].str.lower().str.strip()

Membersihkan Angka (Harga, Rating, Size)

A. Rating & Jumlah Rating:

In [ ]:
# Bersihkan rating_num (pastikan jadi float)
df['rating_num'] = pd.to_numeric(df['rating_num'], errors='coerce')

# Bersihkan noofratings (ada yang berformat "105,991 ratings")
# hapus teks 'ratings', hapus koma, lalu ubah jadi float
df['noofratings'] = df['noofratings'].astype(str).str.replace(' ratings', '', regex=False)
df['noofratings'] = df['noofratings'].str.replace(',', '', regex=False)
df['noofratings'] = pd.to_numeric(df['noofratings'], errors='coerce')

# Isi rating yang kosong (NaN) dengan 0 atau rata-rata
df['rating_num'] = df['rating_num'].fillna(0)

B. Harga (prize)

In [ ]:
# Ubah price_num jadi float
df['price_num'] = pd.to_numeric(df['price_num'], errors='coerce')

# Opsi 1: Hapus baris yang harganya NaN (kalau mau data bersih total)
# df = df.dropna(subset=['price_num'])

# Opsi 2: Isi harga NaN dengan median per kategori (lebih aman buat analisis)
# df['price_num'] = df.groupby('subcategory')['price_num'].transform(lambda x: x.fillna(x.median()))

C. Kolom Size (ML / Gram):

In [ ]:
# Ubah size_value jadi float
df['size_value'] = pd.to_numeric(df['size_value'], errors='coerce')
df['size_ml_est'] = pd.to_numeric(df['size_ml_est'], errors='coerce')
df['size_g_est'] = pd.to_numeric(df['size_g_est'], errors='coerce')

In [ ]:
# 1. Ubah semua nama kolom jadi huruf kecil dan ganti spasi jadi underscore
df.columns = df.columns.str.lower().str.replace(' ', '_')

# 2. Hapus karakter yang bikin error (selain huruf, angka, dan underscore)
# Ini akan menghapus tanda hubung (-) atau simbol aneh lainnya
df.columns = df.columns.str.replace(r'[^a-z0-9_]', '', regex=True)

# 3. Cek hasilnya
print("Nama kolom baru:", df.columns.tolist())

# 4. Simpan ulang (WAJIB pakai encoding='utf-8')
df.to_csv('cosmetic_clean_ready.csv', index=False, encoding='utf-8')

Nama kolom baru: ['product_name', 'website', 'country', 'category', 'subcategory', 'title_href', 'price', 'brand', 'ingredients', 'form', 'type', 'color', 'size', 'rating', 'noofratings', 'rating_num', 'reviews_count', 'price_num', 'price_outlier_flag', 'size_value', 'size_unit', 'size_ml_est', 'size_g_est', 'product_key', 'has_review', 'data_quality_flag', 'best_value_flag', 'hidden_gem_flag', 'risky_flag', 'premium_flag']


Feature Engineering (Buat Kolom Baru untuk Dashboard)

In [ ]:
# Buat kolom Price Category (Budget, Mid-Range, Premium)
def categorize_price(price):
    if pd.isna(price): return 'Unknown'
    if price < 200: return 'Budget (<200)'
    elif price <= 500: return 'Mid-Range (200-500)'
    else: return 'Premium (>500)'

df['price_category'] = df['price_num'].apply(categorize_price)

# Buat kolom "Popularity" berdasarkan jumlah rating
def categorize_popularity(rating_count):
    if pd.isna(rating_count): return 'Unrated'
    if rating_count < 100: return 'Low Popularity'
    elif rating_count <= 1000: return 'Medium Popularity'
    else: return 'High Popularity'

df['popularity_level'] = df['noofratings'].apply(categorize_popularity)

In [ ]:
import pandas as pd
import re

# 1. Baca ulang file asli (atau gunakan df yang sudah kamu bersihkan)
df = pd.read_csv('cosmetic_clean.csv')

# 2. Fungsi untuk membersihkan nama kolom secara brutal
def clean_column_name(col):
    # Ubah ke huruf kecil
    col = col.lower()
    # Ganti semua karakter yang BUKAN huruf, angka, atau underscore dengan underscore
    col = re.sub(r'[^a-z0-9_]', '_', col)
    # Hapus underscore beruntun (misal jadi __)
    col = re.sub(r'_+', '_', col)
    # Hapus underscore di awal/akhir
    col = col.strip('_')
    return col

# 3. Terapkan ke semua kolom
df.columns = [clean_column_name(c) for c in df.columns]

# 4. Cek hasilnya (pastikan tidak ada spasi atau simbol aneh)
print("Nama kolom baru:", df.columns.tolist())

# 5. Simpan ulang dengan encoding UTF-8 (WAJIB)
df.to_csv('cosmetic_clean_ready.csv', index=False, encoding='utf-8')

Nama kolom baru: ['product_name', 'website', 'country', 'category', 'subcategory', 'title_href', 'price', 'brand', 'ingredients', 'form', 'type', 'color', 'size', 'rating', 'noofratings', 'rating_num', 'reviews_count', 'price_num', 'price_outlier_flag', 'size_value', 'size_unit', 'size_ml_est', 'size_g_est', 'product_key', 'has_review', 'data_quality_flag', 'best_value_flag', 'hidden_gem_flag', 'risky_flag', 'premium_flag']


Cek Duplikat & Simpan

In [ ]:
# Hapus duplikat berdasarkan product_name dan website
df = df.drop_duplicates(subset=['product_name', 'website'])

# Cek summary statistik untuk memastikan tidak ada error
print(df.describe())

# Simpan hasil cleaning ke file CSV baru
df.to_csv('cosmetic_clean_ready.csv', index=False)
print("Data siap di-export!")

              price   rating_num  reviews_count     price_num   size_value  \
count   8672.000000  8557.000000    8826.000000   8671.000000  7143.000000   
mean    2455.153873     4.205774    1339.431113   2455.410494   100.706393   
std     3272.359575     0.393041    7767.428872   3272.461022   186.999467   
min        3.500000     1.700000       4.000000      3.500000     0.000000   
25%      599.000000     4.070000      18.000000    599.000000     5.100000   
50%     1552.300000     4.200000      78.000000   1552.300000    33.900000   
75%     3067.097500     4.500000     500.750000   3070.065000   120.000000   
max    94099.000000     5.000000  220040.000000  94099.000000  7500.000000   

       size_ml_est  size_g_est  
count    69.000000   62.000000  
mean     33.568066   10.745375  
std      68.147332   14.837658  
min       1.182940    0.850485  
25%       7.393375    2.622329  
50%       8.872050    4.535920  
75%      13.012340   10.843684  
max     295.735000   85.048500  
